<a href="https://colab.research.google.com/github/HumzaW245/LabelShiftExperiments/blob/versionA/Trials_H2T_coding_head2toeSetupAttempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***h2t replicating attempt***



# Train Test functionality

In [1]:
import torchvision.models as models

from numpy.random import RandomState
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Subset


from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def train(model, device, train_loader, optimizer, epoch, display=True):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        torch.cuda.empty_cache() # Necessary for efficiency and cuda errors. Maybe even put somewhere in train function for each batch
    if display:
      print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
          epoch, batch_idx * len(data), len(train_loader.dataset),
          100. * batch_idx / len(train_loader), loss.item()))

def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, size_average=False).item() # sum up batch loss
            pred = output.max(1, keepdim=True)[1] # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()
            torch.cuda.empty_cache() # Necessary for efficiency and cuda errors. Maybe even put somewhere in train function for each batch

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
    return 100. * correct / len(test_loader.dataset)

# Load Data

In [3]:

from torchvision import datasets,transforms
import torch
import numpy as np


import torchvision.transforms as transforms
from torchvision.datasets import SVHN

# Define the transforms to apply (--------------------------------------PREPROCESSING FOR EACH DATASET WHAT IS BEST TO GO WITH IMAGENETR50)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create an instance of the SVHN dataset with the transforms
trainData = datasets.SVHN('../data', split='train', download=True, transform=transform)

testData = datasets.SVHN('../data', split='test', download=True, transform=transform)


train_loader = torch.utils.data.DataLoader(trainData,
                                           batch_size=32,
                                           shuffle=True,
                                           drop_last=True)

test_dataloader = torch.utils.data.DataLoader(testData,
                                          batch_size=32,
                                          shuffle=True,
                                          drop_last=True) #Drop last is really just to make sure if last batch is not of equal size, drop it. Nothing to do with setting it aside for testing


100%|██████████| 182040794/182040794 [00:08<00:00, 20373153.99it/s]


100%|██████████| 64275384/64275384 [00:05<00:00, 11174618.57it/s]


# Model

In [4]:
def numUniqueClasses(datasetName):

  datasetsClasses = {'SVHN': 10}
  print(f'dataset {datasetName} has {datasetsClasses[datasetName]} unique classes')
  return datasetsClasses[datasetName]

def freezeBackbone(backbone):
  for i, param in enumerate(backbone.parameters()):
    param.requires_grad = False

# model = models.resnet50(pretrained=True)
# freezeBackbone(model)

# print(model)

In [5]:

from typing import Dict, Iterable, Callable
from torch import Tensor

class FeatureExtractor(nn.Module):
    def __init__(self, model: nn.Module, layers: Iterable[str]):
        super().__init__()
        self.model = model
        self.layers = layers
        self._features = {layer: torch.empty(0) for layer in layers}

        for layer_id in layers:
            layer = dict([*self.model.named_modules()])[layer_id]
            layer.register_forward_hook(self.save_outputs_hook(layer_id))

    def save_outputs_hook(self, layer_id: str) -> Callable:
        def fn(_, __, output):
            self._features[layer_id] = output
        return fn

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        _ = self.model(x)
        return self._features

In [18]:
'''
See example 2: https://medium.com/the-dl/how-to-use-pytorch-hooks-5041d777f904

for reference on using forward hooks to get intermediate outputs

'''

from typing import Dict, Iterable, Callable
from torch import Tensor
class Net(torch.nn.Module):
    def __init__(self, datasetName, finetune_backbone, targetSize, layers: Iterable[str]):
        super(Net, self).__init__()
        self.model = models.resnet50(pretrained=True)

        #in_features = self.model.fc.in_features #The fc layer of resenet50 is Linear(in_features=2048, out_features=1000, bias=True) so storing the 2048 and replacing this to map from 2048 to numClasses for target task ====can see the fc layer like this: backbone = models.resnet50(pretrained=True) => print(backbone.fc)

        self.targetSize = targetSize #See comment above...2048 for now used since testing with fc layer as concatenated layer (The fc layer of resenet50 is Linear(in_features=2048, out_features=1000, bias=True))

        self.model.fc = nn.Identity()  # Replace the classifier layer with Identity since classifier will be separately applied after features chosen are extracted (See forward function)

        self.finetune_backbone = finetune_backbone
        if(self.finetune_backbone == False):
          freezeBackbone(self.model)





        # Apply adaptive pooling to resize the tensor
        self.adaptive_pool = nn.AdaptiveAvgPool1d(self.targetSize)

        #New output head
        targetTaskOutFeatures = numUniqueClasses(datasetName) # num of classes in target task
        self.newOutputHead = nn.Linear(self.targetSize, targetTaskOutFeatures, bias=True)  # Create a new classifier


        #Forward hook setup to store intermediate outputs of chosen layers/features
        self.layers = layers
        self._layersChosen = {}

        for name, module in self.model.named_modules():
            if name in self.layers:
              #print(f'layer name is {name}')
              module.register_forward_hook(self.save_outputs_hook(name)) #Name is what the layer_id is in save_outputs_hook...usually forward hooks are not callable so just have (Self, module, input, output) but here a function with those is defined so it can call with passed argument

    def forward(self, x):
        fwdPassBeforeClassifier = self.model(x)
        #print(fwdPassBeforeClassifier.shape)
        selected_features = self._layersChosen # At this point, have not gone through classifier but have all chosen features so can now pass this through a linear layer for classification (FIRST NEED TO CONCAT etc and make it passable to linear layers)
        #print(selected_features)


        #Concatenated Layer for classifier
        concatenated_features = self.getConcatenatedLayer(selected_features) #This is flattening everything passed starting from dim 1 (see definition)


        x = self.newOutputHead(concatenated_features)

        return x

    '''
    Forward hooks (Basically telling upon initialization to keep track of modules specified with module.register_forward_hook) - The values being tracked are updated as forward passes are done and since we append those 'variables' to self._layersChosen, can access it always. (So after self.model(x), their values are updated and so self._layersChosen will have the updated values which we can use to build the new concatenated layer)

    ***********CARE: Make sure:
          IF using a LIST to append outputs: Clear the list each time forward pass is called so not just appending same features over and over again. AND CHANGE  getConcatenatedLayer() function to work with list

          OR

          Use a dictionary and update value at specific keys.AND CHANGE  getConcatenatedLayer() function to work with dict

    '''
    def save_outputs_hook(self, layer_id: str) -> Callable:
        def fn(module, input, output):
            self._layersChosen[layer_id] = output            # Can use this if want to store the name passed in a dictionary with key = name value = output
            #print("Appending to layers chosen list")
            #self._layersChosen.append(output)
        return fn


    '''
    Process of concatenating layers.
    **Similar to flatten_and_concat function of head2toe repo *************************

    NOTE: In head2toe paper, section 3.3, figure 5, pooling is done first and then flatten->normalize.
    BUT, if do that, figure out how will the target size will be achieved  since each tensor in the list for tensors to concatenate of features will be of different size initially

      -============================MIGHT NEED TO CORRECT THIS -> See section 3.3 of paper and flatten_and_concat in finetune.py of h2t repo===========================
                                    NOT Adaptive pooling since that immediately reduces size of Wall to W_targetsize since concatenated layer is reduced to target size
                                    Instead want to do 1-D Strided pooling when shape is [x, x, x] and
                                    2-D Strided pooling when shape is [x, x, x, x]
                                    and NO pooling(just append) if shape is [x, x] (e.g. 'fc' layer output ...even if identity, it will have previous result at that size)

                                    -----SEE if len(output.shape) == 4:, elif, elif, logic in flatten_and_concat AND
                                    See how appending on all_features is happening.

      =========================================
      =========================================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      ============CORRECTION NEEDED...SEE ABOVE=============================
      =========================================
      =========================================
      =========================================
      =========================================
      =========================================


    1) Flatten all tensor to shape [batch_size, features]

    2) Apply adaptive pooling to match target size

    3) Normalize


    This is flattening everything passed starting from dim 1
    e.g.
    "layer4" was stored in list of selected_features
    Shape of feature before flattening: torch.Size([32, 2048, 7, 7])
    Shape of feature after flattening: torch.Size([32, 100352])

    "fc" was stored in list of selected_features
    Shape of feature before flattening: torch.Size([32, 2048])
    Shape of feature after flattening: torch.Size([32, 2048])

    Final concatenated shape:
    Shape of final concatenated layer torch.Size([32, 102400])
    '''
    def getConcatenatedLayer(self, selected_features):
      flattened_tensors = []
      for key, tensor in selected_features.items():
        #print(f'Shape of feature before flattening: {tensor.shape}')
        flatTensor = torch.flatten(tensor, 1)

        #print(f'Shape of feature after flattening: {flatTensor.shape}')
        flattened_tensors.append(flatTensor)

      concatenatedLayer = torch.cat(flattened_tensors, dim=1) #Concatenating flattened layers
      #print(f'Shape of concatenated layer {concatenatedLayer.shape}')


      #Apply adaptive pooling to resize the tensor
      pooled_concatenatedLayer = self.adaptive_pool(concatenatedLayer)  #Pass through adaptive pooling layer to change size to target size


      #Normalize
      final_concatenatedLayer = torch.nn.functional.normalize(pooled_concatenatedLayer, p=2, dim=1)

      #print(f'AFTER Adaptive pooling and normalization, shape of final concatenated layer {final_concatenatedLayer.shape}')
      return final_concatenatedLayer

# Run experiment

In [16]:
def evaluate(datasetName, finetune_backbone=False):
  use_cuda = torch.cuda.is_available()
  device = torch.device("cuda" if use_cuda else "cpu")
  print(device) # you will really need gpu's for this part




  accs = []


  # Load the pre-trained ResNet-50 model
  model = Net(datasetName, finetune_backbone, 2048, layers=["layer4", "fc"])




  optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

  model.to(device)
  for epoch in range(2):
    train(model, device, train_loader, optimizer, epoch, display=True)

  accs.append(test(model, device, test_dataloader))

  accs = np.array(accs)
  print('Acc over 1 instances: %.2f +- %.2f'%(accs.mean(),accs.std()))

In [17]:
evaluate('SVHN', False)

cpu
dataset SVHN has 10 unique classes
Shape of pooled layer 0.4559837281703949
Shape of normalized layer 0.018882766366004944
Shape of pooled layer 0.4571428894996643
Shape of normalized layer 0.018887288868427277


KeyboardInterrupt: ignored

# TODO:

[x]- Transfer to vscode as is IN CLUSTER +++++ GIT REPO

[x]- RUN TO REPRODUCE ABOVE RESULTS FIRST

[x]- Once working, check FT also working

[x]- Then, rework functions to use a config file so cleaner and easier to change stuff...

[x] setup wandb

[x] Reproduce results to best ability for 4 datasets already in wandb.

- Once cleanly setup, implement basic head2toe functionality in separate file
  ****************HARD CODE AS MUCH AS NEEDED TO GET GENERAL H2T functionality tested...can define functions in detail later...FIRST JUST SETUP RANDOM FRACTION OF FEATURES TO GRAB or indices of neurons or w.e....FOCUS ON JUST GETTING SOME TRAINING DONE WITH INTERMEDIATE NEURONS BEING USED TO PASS TO CLASSIFIER (or whatever h2t does exactly)...HARDCODE STUFF AS NEEDED

  ===================================FOR H2T

  - Can just have all layers concatenated to train the classifier (like paper says... basically like giving Wall compatible x to the Linear layer)

  - Above will get us the important features
  
  - then can have another phase that takes the indices of those features so only those are saved and concatenated for actual head2toe eval...

  - I thinkkkkkkkkkkkkkkkkkkkk essentially indices are the same for second phase always so see if any way to get from existing h2t repo ORRRR make sure to save the indices for all future runs... DO NOT SPEND TIME RUNNING FIRST PHASE OF GETTING INDICES EACH TIME IF ALWAYS GONNA BE SAME VALUES THAT ARE MOST IMPORTANT FEATURES



= SEE TRAIN/TEST FUNCTIONS...UPDATE WHEN CEDAR BACK LIVE TO USE CLEAR CACHE  LINE... MAKE SURE STILL LEARNING THO